### Text Embedding
Pr-train BERT model

In [4]:
from transformers import  AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
model = AutoModel.from_pretrained("BAAI/bge-m3")

input_text = "Hello, my dog is cute"
inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True )
with torch.no_grad():
    embeddings = model(**inputs).last_hidden_state.mean(dim=1)  
print(embeddings)

c:\RAG_POC\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rohit\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 17938.88it/s]


tensor([[-1.0732, -0.0650, -0.8150,  ..., -0.0026, -0.4978,  0.4431]])


In [ ]:
import base64
import io
import json
import os
import warnings
from pathlib import Path

import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer

try:
    from rapidocr_onnxruntime import RapidOCR
except Exception:
    RapidOCR = None
    warnings.warn("rapidocr_onnxruntime not available; OCR will be skipped")

IN_JSONL = Path("../data/processed/relay_manual.chunks.jsonl")
MODEL_NAME = "BAAI/bge-m3"

if not IN_JSONL.exists():
    raise SystemExit(f"Input JSONL not found: {IN_JSONL}")

print(f"Loading local embedding model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
OCR_ENGINE = RapidOCR() if RapidOCR else None


def ocr_image_base64(base64_str: str) -> str:
    if not base64_str or OCR_ENGINE is None:
        return ""
    try:
        image_bytes = base64.b64decode(base64_str)
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        results, _ = OCR_ENGINE(np.array(image))
        if not results:
            return ""
        texts = []
        for item in results:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                text = item[1]
                if isinstance(text, str) and text.strip():
                    texts.append(text.strip())
        return "\n".join(texts)
    except Exception as exc:
        print(f"OCR skipped for one image: {exc}")
        return ""


def build_embed_text(record: dict) -> str:
    parts = []

    for value in [record.get("embed_text"), record.get("text"), record.get("section")]:
        if value:
            parts.append(str(value))

    headings = record.get("headings") or []
    if headings:
        parts.append("Headings: " + " | ".join([str(h) for h in headings if h]))

    for image in record.get("images") or []:
        image_label = image.get("image_id") or "image"
        ocr_text = ocr_image_base64(image.get("base64", ""))
        if ocr_text.strip():
            parts.append(f"[OCR_IMAGE_TEXT:{image_label}]\n" + ocr_text)
            # print(f"Image {image_label}: found image text")
            # print(ocr_text)
        else:
            parts.append(f"[OCR_IMAGE_TEXT:{image_label}]\nNo text found")
    print(parts)
    return "\n\n".join(parts)


def embed_texts(texts):
    embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    return embeddings


def all_embeddings():
    with IN_JSONL.open("r", encoding="utf-8") as fh:
        records = [json.loads(line) for line in fh if line.strip()]

    print(f"Loaded {len(records)} chunk records from {IN_JSONL}")

    texts = []
    for r in records:
        texts.append(build_embed_text(r))

    embeddings = embed_texts(texts)
    print("\nEmbedding model:", MODEL_NAME)
    print("Embedding count:", len(embeddings))
    print("Embedding shape:", embeddings.shape)
    print("First embedding vector sample:", embeddings[0][:10].tolist())
    print("Done.")
    return embeddings , texts


if __name__ == "__main__":
    all_embeddings()


## Data Vectorization

In [11]:
import psycopg2
from pgvector.psycopg2 import register_vector

HOST = "localhost"
PORT = 5434
USER = "admin"
PASSWORD = "pass123"
DATABASE = "RAG_POC"
TABLE = "rag_chunks"
SOURCE_FILE = "ABB 800xA.pdf"


def create_database():
    conn = None
    try:
        # Connect to the default 'postgres' database
        conn = psycopg2.connect(
            host=HOST,
            port=PORT,
            user=USER,
            password=PASSWORD,
            dbname="postgres"
        )
        conn.autocommit = True
        cur = conn.cursor()

        # Check if the target database exists
        cur.execute(
            "SELECT 1 FROM pg_database WHERE datname = %s",
            (DATABASE,)
        )

        if cur.fetchone() is None:
            print(f"Creating database: {DATABASE}")
            cur.execute(f'CREATE DATABASE "{DATABASE}"')
        else:
            print(f"Database already exists: {DATABASE}")

        cur.close()
        conn.close()

    except Exception as e:
        print(f"Error while creating database: {e}")


def create_table():
    conn = None
    try:
        conn = psycopg2.connect(
            host=HOST,
            port=PORT,
            user=USER,
            password=PASSWORD,
            dbname=DATABASE
        )
        register_vector(conn)
        cur = conn.cursor()

        # Enable pgvector extension inside RAG_POC
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector")

        cur.execute(f"""
            CREATE TABLE IF NOT EXISTS {TABLE} (
                chunk_id UUID PRIMARY KEY,
                source TEXT,
                page_content TEXT,
                embedding VECTOR(1024),
                metadata JSONB,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)

        conn.commit()
        print("Table ready.")

        cur.close()
        conn.close()

    except Exception as e:
        print(f"Error while creating table: {e}")

if __name__ == "__main__":
    create_database()
    create_table()

Database already exists: RAG_POC
Table ready.


## Vectorize Data

In [13]:
# ------------------------------------------------------------------------------
# Database
# ------------------------------------------------------------------------------
import uuid
conn = None

conn = psycopg2.connect(
    host=HOST,
    port=PORT,
    user=USER,
    password=PASSWORD,
    dbname=DATABASE
)
register_vector(conn)
cur = conn.cursor()

embeddings, texts = all_embeddings()

for text, emb in zip(texts, embeddings):

    chunk_id = str(uuid.uuid4())

    metadata = {
        "chunk_id": chunk_id,
        "source": SOURCE_FILE, 
    }

    cur.execute(
        f"""
        INSERT INTO {TABLE}
        (
            chunk_id,
            source,
            page_content,
            embedding,
            metadata
        )
        VALUES (%s,%s,%s,%s,%s)
        """,
        (
            chunk_id,
            SOURCE_FILE,
            text,
            emb,
            json.dumps(metadata)
        )
    )

conn.commit()

cur.close()
conn.close()

print("Done.")
print("Stored", len(embeddings), "embeddings.")
print("Stored", len(texts), "text.")

Loaded 321 chunk records from ..\data\processed\relay_manual.chunks.jsonl
['Image', 'Image', '[OCR_IMAGE_TEXT:img_00001]\n0.bq35']
['Power and productivity for a better world TM\nSystem Version 5.1\nImage\nSystem Version 5.1', 'Power and productivity for a better world TM\nSystem Version 5.1\nImage\nSystem Version 5.1', 'System 800xA Operations', 'Headings: System 800xA Operations', '[OCR_IMAGE_TEXT:img_00002]\nNo text found']
['This document contains information about one or more ABB products and may include a description of or a reference to one or more standards that may be generally relevant to the ABB products. The presence of any such description of a standard or reference to a standard is not a representation that all of the ABB products referenced in this document support all of the features of the described or referenced standard. In order to determine the specific features supported by a particular ABB product, the reader should consult the product specifications for the part

Batches: 100%|██████████| 11/11 [07:58<00:00, 43.51s/it]



Embedding model: BAAI/bge-m3
Embedding count: 321
Embedding shape: (321, 1024)
First embedding vector sample: [-0.04240307956933975, 0.026067472994327545, -0.040832825005054474, -0.053011879324913025, -0.019597571343183517, 5.0916562031488866e-05, 0.003926129080355167, 0.019240226596593857, 0.00276371487416327, -0.023170659318566322]
Done.
Done.
Stored 321 embeddings.
Stored 321 text.


In [21]:
import psycopg2
from pgvector.psycopg2 import register_vector

conn = psycopg2.connect(
    host=HOST,
    port=PORT,
    user=USER,
    password=PASSWORD,
    dbname=DATABASE
)

register_vector(conn)

cur = conn.cursor()

chunk_id = "cf5cd03c-cab3-4dd0-85cf-ac229061188b"

cur.execute(
    """
    SELECT page_content
    FROM rag_chunks
    WHERE chunk_id = %s;
    """,
    (chunk_id,)
)

row = cur.fetchone()

if row:
    text = row[0]
    print("Text Length:", len(text))
    print(text)
else:
    print("No record found.")

cur.close()
conn.close()

Text Length: 8098
Section 11 - Structured Data Logger, = . SDL Data View..............................................................................................................239, = . Appendix A - System Alarm Messages, = . Operations......................................................................................................................242, = . Device Management Foundation Fieldbus ....................................................................245, = . Batch Management........................................................................................................246, = . 800xA History, = ...............................................................................................................247. PC, Network Software and Monitoring (PNSM), = ..........................................................248. 800xA for Advant Master..............................................................................................250, = . Melody..............

In [ ]:
import os
import json
from typing import List, Dict, Any, Optional

import numpy as np
import psycopg2
from psycopg2.extras import Json, execute_values
from pgvector.psycopg2 import register_vector

# Default connection parameters (override with env vars)
DB_HOST = os.getenv("PGHOST", "localhost")
DB_PORT = int(os.getenv("PGPORT", "5434"))  # using port 5434 per your message
DB_NAME = os.getenv("PGDATABASE", "postgres")
DB_USER = os.getenv("PGUSER", "admin")
DB_PASSWORD = os.getenv("PGPASSWORD", "pass123")

TABLE_NAME = os.getenv("PGVECTOR_TABLE", "embeddings")
VECTOR_COLUMN = "embedding"
ID_COLUMN = "id"
TEXT_COLUMN = "text"
META_COLUMN = "metadata"

def connect():
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
    )
    # register pgvector adapter for psycopg2
    register_vector(conn)
    return conn

def create_extension_if_needed(conn):
    with conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    conn.commit()

In [17]:
def create_table_if_not_exists(conn, dim: int):
    """
    Create table with a pgvector column of given dimension.
    """
    with conn.cursor() as cur:
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
            {ID_COLUMN} TEXT PRIMARY KEY,
            {TEXT_COLUMN} TEXT,
            {META_COLUMN} JSONB,
            {VECTOR_COLUMN} vector({dim})
        );
        """
        cur.execute(create_sql)
        # create an index for faster similarity search (optional: ivfflat requires REINDEX/ANALYZE)
        cur.execute(
            f"CREATE INDEX IF NOT EXISTS {TABLE_NAME}_vector_idx ON {TABLE_NAME} USING ivfflat ({VECTOR_COLUMN}) WITH (lists = 100);"
        )
    conn.commit()

In [ ]:

def upsert_embeddings(conn, embeddings: np.ndarray, records: List[Dict[str, Any]], id_field: str = "chunk_id"):
    """
    Upsert embeddings into the DB.
    embeddings: numpy array shape (N, dim)
    records: list of dicts (same length as embeddings). Each record should contain an id (id_field) and optional text/metadata.
    id_field: key in record to use as primary id. If not present, uses index-based id.
    """
    if len(embeddings) != len(records):
        raise ValueError("embeddings and records must have the same length")

    # Prepare rows for bulk upsert
    rows = []
    for i, (vec, rec) in enumerate(zip(embeddings, records)):
        # ensure vector is a Python list of floats
        vec_list = [float(x) for x in np.asarray(vec).tolist()]
        rec_id = str(rec.get(id_field, f"row_{i}"))
        text = rec.get("text") or rec.get("embed_text") or rec.get("OCR_IMAGE_TEXT")
        metadata = rec.copy()
        # remove large fields if desired
        metadata.pop("text", None)
        metadata.pop("embed_text", None)
        metadata.pop("OCR_IMAGE_TEXT", None)
        rows.append((rec_id, text, Json(metadata), vec_list))

    with conn.cursor() as cur:
        # Use INSERT ... ON CONFLICT to upsert
        sql = f"""
        INSERT INTO {TABLE_NAME} ({ID_COLUMN}, {TEXT_COLUMN}, {META_COLUMN}, {VECTOR_COLUMN})
        VALUES %s
        ON CONFLICT ({ID_COLUMN}) DO UPDATE
        SET
          {TEXT_COLUMN} = EXCLUDED.{TEXT_COLUMN},
          {META_COLUMN} = EXCLUDED.{META_COLUMN},
          {VECTOR_COLUMN} = EXCLUDED.{VECTOR_COLUMN};
        """
        # execute_values will adapt the Python list to the vector type via pgvector adapter
        execute_values(cur, sql, rows, template="(%s, %s, %s, %s)")
    conn.commit()